## EDA - Flights and Weather Data at JFK airport - Analysing Sandy Hurrican 2011 

In [1]:
## import dependencies

# Pandas for manipulation
import pandas as pd 
# SQLalchemy for connecting with SQL database and writing SQL queries
from sqlalchemy import create_engine, text 
# dotent to deal with credentials 
from dotenv import dotenv_values

In [2]:
## Connect with database
config = dotenv_values()

db_user   = config['POSTGRES_USER']
db_pw     = config['POSTGRES_PASS']
db_host   = config['POSTGRES_HOST']
db_port   = config['POSTGRES_PORT']
db_db     = config['POSTGRES_DB']
db_schema = config['POSTGRES_SCHEMA']
db        = 'postgresql'

In [3]:
## DATABASE URL -> PostgreSQL connection URL 
URL = f'{db}://{db_user}:{db_pw}@{db_host}:{db_port}/{db_db}'

In [4]:
## Create an engine with connection URL 
engine = create_engine(URL, echo=True)

In [5]:
## Set the schema search path 
with engine.begin() as con:
    result = con.execute(text(f'SET search_path TO {db_schema};'))
result

2025-09-15 19:07:49,081 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-09-15 19:07:49,082 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-09-15 19:07:49,144 INFO sqlalchemy.engine.Engine select current_schema()
2025-09-15 19:07:49,145 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-09-15 19:07:49,214 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-09-15 19:07:49,215 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-09-15 19:07:49,279 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-09-15 19:07:49,280 INFO sqlalchemy.engine.Engine SET search_path TO clear_skies;
2025-09-15 19:07:49,281 INFO sqlalchemy.engine.Engine [generated in 0.00039s] {}
2025-09-15 19:07:49,334 INFO sqlalchemy.engine.Engine COMMIT


In [6]:
## Create a function to query 
def query_to_df(query: str)-> pd.DataFrame:
    '''
    This function takes a PostgreSQL Qurey and returns 
    a dataframe from the queried tabel. 

    NOTE: for this function to run you need Pandas and already a valid connection 
    to your DB with SQLalchemy
    '''
    with engine.begin() as con:
        call = con.execute(text(query))
    data = call.all()
    df = pd.DataFrame(data)

    return df

In [51]:
## FROM FLIGHTS FROM JFK OR TO 
df_flights = query_to_df(
    '''
        SELECT
            *
        FROM prep_flights
    '''
)

2025-09-15 20:26:00,748 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-09-15 20:26:00,753 INFO sqlalchemy.engine.Engine 
        SELECT
            *
        FROM prep_flights
    
2025-09-15 20:26:00,753 INFO sqlalchemy.engine.Engine [generated in 0.00157s] {}
2025-09-15 20:26:01,662 INFO sqlalchemy.engine.Engine COMMIT


In [7]:
df_weather = query_to_df(
    '''
        SELECT
            airport_code, 
            station_id,
            CAST(date AS DATE) AS sensor_date,
            date_year,
            month_name,
            date_day,
            avg_temp_c,
            min_temp_c,
            max_temp_c,
            max_snow_mm,
            avg_wind_speed,
            precipitation_mm,
            avg_pressure_hpa
        FROM prep_weather_daily
    '''
)

2025-09-15 19:08:00,041 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-09-15 19:08:00,042 INFO sqlalchemy.engine.Engine 
        SELECT
            airport_code, 
            station_id,
            CAST(date AS DATE) AS sensor_date,
            date_year,
            month_name,
            date_day,
            avg_temp_c,
            min_temp_c,
            max_temp_c,
            max_snow_mm,
            avg_wind_speed,
            precipitation_mm,
            avg_pressure_hpa
        FROM prep_weather_daily
    
2025-09-15 19:08:00,043 INFO sqlalchemy.engine.Engine [generated in 0.00075s] {}
2025-09-15 19:08:00,151 INFO sqlalchemy.engine.Engine COMMIT


In [52]:
print(f'Daily Weather data: {len(df_weather)}')
print(f'Daily flights data: {len(df_flights)}')

Daily Weather data: 183
Daily flights data: 108694


Flight data are much much abundant because there are multiple flights per day, whereas daily weather data consist of one aggregated observation per day.

## EDA STEP 01 - Weather Data

In [11]:
df_weather.head()

,airport_code,station_id,sensor_date,date_year,month_name,date_day,avg_temp_c,min_temp_c,max_temp_c,max_snow_mm,avg_wind_speed,precipitation_mm,avg_pressure_hpa
0,LGA,72503,2012-10-01,2012.0,October,1.0,17.8,12.8,22.2,0,17.6,0.0,1011.6
1,EWR,72502,2012-10-01,2012.0,October,1.0,16.6,10.0,22.8,0,14.8,0.0,1011.9
2,JFK,74486,2012-10-01,2012.0,October,1.0,17.1,12.2,22.2,0,18.4,0.0,1012.2
3,JFK,74486,2012-10-02,2012.0,October,2.0,18.8,17.2,21.7,0,9.4,8.4,1015.8
4,LGA,72503,2012-10-02,2012.0,October,2.0,18.8,17.2,21.1,0,10.8,9.9,1015.4


In [49]:
df_weather.tail()

,airport_code,station_id,sensor_date,date_year,month_name,date_day,avg_temp_c,min_temp_c,max_temp_c,max_snow_mm,avg_wind_speed,precipitation_mm,avg_pressure_hpa
178,EWR,72502,2012-11-29,2012,November,29,2.9,-1.7,7.2,0.0,16.9,0.0,1026.6
179,LGA,72503,2012-11-29,2012,November,29,4.9,2.8,6.7,0.0,20.9,0.0,1026.2
180,LGA,72503,2012-11-30,2012,November,30,5.8,3.9,8.3,0.0,14.4,0.0,1028.6
181,EWR,72502,2012-11-30,2012,November,30,4.6,-1.1,8.9,0.0,9.0,0.0,1028.6
182,JFK,74486,2012-11-30,2012,November,30,4.6,0.6,8.3,0.0,13.0,0.0,1029.0


In [12]:
## Check Data Types and nulls
df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183 entries, 0 to 182
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   airport_code      183 non-null    object 
 1   station_id        183 non-null    int64  
 2   sensor_date       183 non-null    object 
 3   date_year         183 non-null    float64
 4   month_name        183 non-null    object 
 5   date_day          183 non-null    float64
 6   avg_temp_c        183 non-null    object 
 7   min_temp_c        183 non-null    object 
 8   max_temp_c        183 non-null    object 
 9   max_snow_mm       183 non-null    int64  
 10  avg_wind_speed    183 non-null    object 
 11  precipitation_mm  183 non-null    object 
 12  avg_pressure_hpa  180 non-null    object 
dtypes: float64(2), int64(2), object(9)
memory usage: 18.7+ KB


- sensor date should be converted into pandas date data type (NOTE: the object is datetime.date but because I am working with pandas I converting to pandas datetime64) 
- date_day should be int 
- avg_temo_c, min_temp_c, max_temp_c, avg_wind_speed, precipitation_mm, avg_pressura_hpa should be floats

In [13]:
## converting data types to be applicable for pandas
## Converts to date 
def to_date(df: pd.DataFrame, col_name: str):
    '''
        Converts a column to date
    '''
    df[col_name] = pd.to_datetime(df[col_name])

## convert a column to flaot data type 
def to_float(df: pd.DataFrame, col_name: str):
    '''
        Converts a column to float 
    '''
    df[col_name] = df[col_name].astype(float)

## convert a column to int data type 
def to_int(df: pd.DataFrame, col_name: str):
    '''
        Converts a column to float
    '''
    df[col_name] = df[col_name].astype(int)

In [14]:
## Converting the data 
to_date(df_weather, 'sensor_date')

to_float(df_weather, 'avg_temp_c')
to_float(df_weather, 'min_temp_c')
to_float(df_weather, 'max_temp_c')
to_float(df_weather, 'max_snow_mm')
to_float(df_weather, 'avg_wind_speed')
to_float(df_weather, 'precipitation_mm')
to_float(df_weather, 'avg_pressure_hpa')

to_int(df_weather, 'date_day')
to_int(df_weather, 'date_year')

In [15]:
df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183 entries, 0 to 182
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   airport_code      183 non-null    object        
 1   station_id        183 non-null    int64         
 2   sensor_date       183 non-null    datetime64[ns]
 3   date_year         183 non-null    int64         
 4   month_name        183 non-null    object        
 5   date_day          183 non-null    int64         
 6   avg_temp_c        183 non-null    float64       
 7   min_temp_c        183 non-null    float64       
 8   max_temp_c        183 non-null    float64       
 9   max_snow_mm       183 non-null    float64       
 10  avg_wind_speed    183 non-null    float64       
 11  precipitation_mm  183 non-null    float64       
 12  avg_pressure_hpa  180 non-null    float64       
dtypes: datetime64[ns](1), float64(7), int64(3), object(2)
memory usage: 18.7+ KB


In [44]:
weather_data_nulls = df_weather.isna().sum()
weather_dublicated = df_weather.duplicated().sum()
print(weather_data_nulls)
print('\n')
print('Count duplicats',weather_dublicated)

airport_code        0
station_id          0
sensor_date         0
date_year           0
month_name          0
date_day            0
avg_temp_c          0
min_temp_c          0
max_temp_c          0
max_snow_mm         0
avg_wind_speed      0
precipitation_mm    0
avg_pressure_hpa    3
dtype: int64


Count duplicats 0


In [ ]:
## -> TODO: check the 3 missing values in avg_pressure_hpa

In [62]:
df_weather.describe()

,station_id,sensor_date,date_year,date_day,avg_temp_c,min_temp_c,max_temp_c,max_snow_mm,avg_wind_speed,precipitation_mm,avg_pressure_hpa
count,183.000000,183,183.0,183.000000,183.000000,183.000000,183.000000,183.000000,183.000000,183.000000,180.000000
mean,73163.666667,2012-10-31 00:00:00,2012.0,15.754098,10.955738,7.416940,14.537158,0.174863,16.496175,1.777049,1018.040556
min,72502.000000,2012-10-01 00:00:00,2012.0,1.000000,2.100000,-2.800000,4.400000,0.000000,4.000000,0.000000,984.900000
25%,72502.000000,2012-10-16 00:00:00,2012.0,8.000000,6.350000,2.800000,10.000000,0.000000,10.800000,0.000000,1012.475000
50%,72503.000000,2012-10-31 00:00:00,2012.0,16.000000,10.200000,6.100000,13.900000,0.000000,15.100000,0.000000,1018.050000
75%,74486.000000,2012-11-15 00:00:00,2012.0,23.000000,15.100000,11.950000,18.900000,0.000000,20.500000,0.300000,1023.600000
max,74486.000000,2012-11-30 00:00:00,2012.0,31.000000,22.000000,20.000000,27.200000,13.000000,58.300000,26.200000,1037.200000
std,937.596204,NaN,0.0,8.831118,5.277587,5.413718,5.505625,1.276164,8.584079,4.651435,9.176925


## EDA STEP 02 - Flights Data

In [63]:
df_flights.head()

,flight_date,dep_time,sched_dep_time,dep_delay,dep_delay_interval,arr_time,sched_arr_time,arr_delay,arr_delay_interval,airline,...,flight_number,origin,dest,air_time,air_time_interval,actual_elapsed_time,actual_elapsed_time_interval,distance_km,cancelled,diverted
0,2012-10-01,02:17:00,00:55:00,82.0,0 days 01:22:00,07:48:00,06:33:00,75.0,0 days 01:15:00,B6,...,98,DEN,JFK,195.0,0 days 03:15:00,211.0,0 days 03:31:00,2616.79,0,0
1,2012-10-01,04:58:00,05:00:00,-2.0,-1 days +23:58:00,06:36:00,06:48:00,-12.0,-1 days +23:48:00,US,...,1117,EWR,CLT,80.0,0 days 01:20:00,98.0,0 days 01:38:00,851.34,0,0
2,2012-10-01,04:53:00,05:01:00,-8.0,-1 days +23:52:00,08:49:00,08:50:00,-1.0,-1 days +23:59:00,B6,...,738,PSE,JFK,211.0,0 days 03:31:00,236.0,0 days 03:56:00,2602.31,0,0
3,2012-10-01,05:23:00,05:25:00,-2.0,-1 days +23:58:00,06:36:00,06:40:00,-4.0,-1 days +23:56:00,EV,...,4140,BTV,EWR,44.0,0 days 00:44:00,73.0,0 days 01:13:00,428.09,0,0
4,2012-10-01,05:31:00,05:33:00,-2.0,-1 days +23:58:00,08:17:00,08:14:00,3.0,0 days 00:03:00,UA,...,1285,EWR,IAH,205.0,0 days 03:25:00,226.0,0 days 03:46:00,2253.08,0,0


In [ ]:
df_flights.head()
df_flights.columns
drop_cols = [
    'dep_delay_interval', 
    'arr_delay_interval',
    'air_time', 
    'actual_elapsed_time', 'actual_elapsed_time_interval', 'distance_km',
]

df_flights.drop(drop_cols, axis=1, inplace=True)

In [65]:
df_flights.head()

,flight_date,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,airline,tail_number,flight_number,origin,dest,air_time_interval,cancelled,diverted
0,2012-10-01,02:17:00,00:55:00,82.0,07:48:00,06:33:00,75.0,B6,N563JB,98,DEN,JFK,0 days 03:15:00,0,0
1,2012-10-01,04:58:00,05:00:00,-2.0,06:36:00,06:48:00,-12.0,US,N185UW,1117,EWR,CLT,0 days 01:20:00,0,0
2,2012-10-01,04:53:00,05:01:00,-8.0,08:49:00,08:50:00,-1.0,B6,N760JB,738,PSE,JFK,0 days 03:31:00,0,0
3,2012-10-01,05:23:00,05:25:00,-2.0,06:36:00,06:40:00,-4.0,EV,N15574,4140,BTV,EWR,0 days 00:44:00,0,0
4,2012-10-01,05:31:00,05:33:00,-2.0,08:17:00,08:14:00,3.0,UA,N37253,1285,EWR,IAH,0 days 03:25:00,0,0


In [68]:
df_flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108694 entries, 0 to 108693
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype          
---  ------             --------------   -----          
 0   flight_date        108694 non-null  object         
 1   dep_time           100364 non-null  object         
 2   sched_dep_time     108694 non-null  object         
 3   dep_delay          100364 non-null  float64        
 4   arr_time           100237 non-null  object         
 5   sched_arr_time     108694 non-null  object         
 6   arr_delay          100062 non-null  float64        
 7   airline            108694 non-null  object         
 8   tail_number        105610 non-null  object         
 9   flight_number      108694 non-null  int64          
 10  origin             108694 non-null  object         
 11  dest               108694 non-null  object         
 12  air_time_interval  100062 non-null  timedelta64[ns]
 13  cancelled          108694 non

In [66]:
flights_data_nulls = df_flights.isna().sum()
flights_dublicated = df_flights.duplicated().sum()
print(flights_data_nulls)
print('\n')
print('Count duplicats',flights_dublicated)

flight_date             0
dep_time             8330
sched_dep_time          0
dep_delay            8330
arr_time             8457
sched_arr_time          0
arr_delay            8632
airline                 0
tail_number          3084
flight_number           0
origin                  0
dest                    0
air_time_interval    8632
cancelled               0
diverted                0
dtype: int64


Count duplicats 0


In [ ]:
flight_filtered_dates  = df_flights_jfk[(df_flights_jfk['flight_date'] > datetime.datetime(2012,10,20)) & (df_flights_jfk['flight_date'] < datetime.datetime(2012,11,20))]

## Filter the data for a better readability
weather_filtered_dates  = df_weather_jfk[(df_weather_jfk['sensor_date'] > datetime.date(2012,10,20)) & (df_weather_jfk['sensor_date'] < datetime.date(2012,11,20))]

NameError: name 'df_flights_jfk' is not defined

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

In [ ]:
cancelled_per_day = flight_filtered_dates.groupby('flight_date')['cancelled'].sum()

fig, ax = plt.subplots(figsize=(20,5))
ax.bar(cancelled_per_day.index, cancelled_per_day.values)
ax.set_xlabel('Flight Date')
ax.set_ylabel('Number of Cancelled Flights')
ax.set_title('Number of Cancelled Flights per Day')
ax.set_xticks(cancelled_per_day.index)
plt.xticks(rotation=45)
plt.show()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Windspeed', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_filtered_dates['sensor_date'].sort_values(ascending=True), weather_filtered_dates['avg_wind_speed'])
ax[1].set_xticks(weather_filtered_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Wind Speed KmH', fontsize=12)
ax[1].set_title('Average Wind Speed per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
flight_filtered_dates.head()

In [ ]:
weather_filtered_dates.head()

In [ ]:
sum(cancelled_per_day)

In [ ]:
merged_df = pd.merge(
    cancelled_per_day,
    weather_filtered_dates,
    left_on='flight_date',
    right_on='sensor_date',
    how='inner' 
)

In [ ]:
selected_cols = ['cancelled','avg_temp_c','min_temp_c','max_temp_c','max_snow_mm','avg_wind_speed','precipitation_mm','avg_pressure_hpa']
new_df = merged_df[selected_cols].copy()
new_df

In [ ]:
## 2012-10-21 to 2012-11-19	
correlation_matrix = new_df.corr()
correlation_matrix

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Pressure', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_filtered_dates['sensor_date'].sort_values(ascending=True), weather_filtered_dates['avg_pressure_hpa'])
ax[1].set_xticks(weather_filtered_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Average Pressure', fontsize=12)
ax[1].set_title('Average Presure Speed per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Windspeed', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_filtered_dates['sensor_date'].sort_values(ascending=True), weather_filtered_dates['avg_wind_speed'])
ax[1].set_xticks(weather_filtered_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Wind Speed KmH', fontsize=12)
ax[1].set_title('Average Wind Speed per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(20,15))
ax = sns.heatmap(new_df.corr())


In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Snow', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_filtered_dates['sensor_date'].sort_values(ascending=True), weather_filtered_dates['max_snow_mm'])
ax[1].set_xticks(weather_filtered_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Snow mm', fontsize=12)
ax[1].set_title('Max Snow per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
import datetime
flight_focused_dates  = df_flights_jfk[(df_flights_jfk['flight_date'] > datetime.datetime(2012,11,2)) & (df_flights_jfk['flight_date'] < datetime.datetime(2012,11,13))]

## Filter the data for a better readability
weather_focused_dates  = df_weather_jfk[(df_weather_jfk['sensor_date'] > datetime.date(2012,11,2)) & (df_weather_jfk['sensor_date'] < datetime.date(2012,11,13))]

cancelled_per_day = flight_focused_dates.groupby('flight_date')['cancelled'].sum()


In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Snow', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_focused_dates['sensor_date'].sort_values(ascending=True), weather_focused_dates['avg_wind_speed'])
ax[1].set_xticks(weather_focused_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Snow mm', fontsize=12)
ax[1].set_title('Max Snow per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
# 25.10.2012
# 10.11.2012

flight_dates_canc_num = df_flights_jfk[(df_flights_jfk['flight_date'] > datetime.datetime(2012,10,25)) & (df_flights_jfk['flight_date'] < datetime.datetime(2012,11,10))]
flight_dates_canc_num

In [ ]:
len(flight_dates_canc_num)
cancelled_flights = flight_dates_canc_num[flight_dates_canc_num['cancelled'] == 1]
sum_cancelled = sum(cancelled_flights['cancelled'])
sum_cancelled

In [ ]:
len(df_flights_jfk)

In [ ]:
all_flights = len(flight_dates_canc_num)

In [ ]:
percentage_canceled = sum_cancelled/all_flights

In [ ]:
round(percentage_canceled * 100,2)

In [ ]:
sum_cancelled